## carregando imagem

In [ ]:
import cv2 as cv
def mostrar_imagem(img):

    cv.imshow("amostra de imagem",img)
    cv.waitKey(0)
    cv.destroyAllWindows()

In [ ]:
#  importando bibliotecas
import cv2 as cv
import numpy as np

#definindo variaveis
largura=800
altura=600
pts_destino= np.float32([
    [0,0], # canto sup esq
    [largura,0], #cant sup dir
    [largura,altura], # cat in dir
    [0,altura] # cant in esq
])

def padronizar(caminho_imagem):
    '''Leitura e Padronização:
    Leia a imagem original do disco.
    Redimensione a imagem para um tamanho padrão (por exemplo, 800x600) para facilitar o processamento.'''
    #ler imagem
    img = cv.imread(caminho_imagem)
    if img is not None:
        print("Imagem carregada")
        return cv.resize(img,(altura,largura))
    else:
        print("erro ao carregar imagem")

    '''Correção de Perspectiva (Envelopamento):
    - Defina os 4 pontos de origem (os cantos distorcidos da bandeja na foto) e os 4 pontos de destino (os cantos de um retângulo perfeito).
    - Utilize as funções matemáticas do OpenCV para realizar o warp perspective (envelopamento), gerando uma nova imagem onde a bandeja aparece perfeitamente vista de cima.'''

    def ordenar_pontos(pontos):
        '''ordena cantos'''
        rect = np.zeros((4, 2), dtype="float32")
        s = pontos.sum(axis=1)
        rect[0] = pontos[np.argmin(s)]
        rect[2] = pontos[np.argmax(s)]
        diff = np.diff(pontos, axis=1)
        rect[1] = pontos[np.argmin(diff)]
        rect[3] = pontos[np.argmax(diff)]
        return rect

    def detectar_cantos(img_resized):
        '''detecta  os cantos na imagem original'''
        ## escala de cinza
        img_gray = cv.cvtColor(img_resized, cv.COLOR_BGR2GRAY)
        # img_blur
        img_blur = cv.GaussianBlur(img_gray,(11,11),0)
        ## theshold
        _,mask = cv.threshold(img_blur,0,255, cv.THRESH_BINARY + cv.THRESH_OTSU)

        # Encontra o maior contorno (que será a folha de papel)
        contornos, _ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
        contorno_maior = max(contornos, key=cv.contourArea)
        
        perimetro = cv.arcLength(contorno_maior, True)
        aproximacao = cv.approxPolyDP(contorno_maior, 0.02 * perimetro, True)
        
        if len(aproximacao) == 4:
            return ordenar_pontos(aproximacao.reshape(4, 2))
            
        # Fallback se algo der muito errado
        print("Aviso: Papel não detectado. Usando pontos de fallback.")
        return np.float32([[100, 150], [700, 150], [50, 500], [750, 500]])

    def corrigir_perspectiva(self, img_resized, pts_origem):
        # Realiza o warp perspective (envelopamento) com os 4 pontos detectados[cite: 1]
        matriz = cv.getPerspectiveTransform(pts_origem, pts_destino)
        return cv.warpPerspective(img_resized, matriz, (largura, altura))


'Correção de Perspectiva (Envelopamento):\n- Defina os 4 pontos de origem (os cantos distorcidos da bandeja na foto) e os 4 pontos de destino (os cantos de um retângulo perfeito).\n- Utilize as funções matemáticas do OpenCV para realizar o warp perspective (envelopamento), gerando uma nova imagem onde a bandeja aparece perfeitamente vista de cima.'

In [14]:
caminho_imagem="data/raw/Exer_1.jpeg"

img_padronizada=padronizar(caminho_imagem)


Imagem carregada
